# OpenPOPCON CENTAUR example

CENTAUR is a compact, high-field, **negative triangularity** breakeven
tokamak: 2.0 m major radius, 10.9 T, delta = -0.55, designed for 40 MW of
fusion power at Q = 1.3 in a 10 s pulse. Parameters are from Table 1 of
[arXiv:2605.27549](https://arxiv.org/abs/2605.27549).

Because it is a negative triangularity device, this example uses the
`H_NT23` confinement scaling rather than `H98y2`.

In [ ]:
import numpy as np
import openpopcon as op

## Setup and run

In [ ]:
settingsfile = "./POPCON_input_example.yaml"
plotsettingsfile = "./plotsettings.yml"

pc = op.POPCON(settingsfile=settingsfile, plotsettingsfile=plotsettingsfile)
pc.run_POPCON()

## Plotting

The grey region is where the balance has no physical solution, here because
impurity radiation exceeds the confinement loss. CENTAUR is a marginal,
breakeven-class device, so that boundary sits close to the operating point
and a good part of the low-temperature grid is inaccessible.

In [ ]:
fig, ax = pc.plot()

### Checking against the design point

The published operating point is 0.55 of the Greenwald density at a
volume-averaged ion temperature of 5.15 keV. The profile shapes in the
settings file were chosen so the fusion power there matches the published
40 MW.

Read the note at the bottom of `POPCON_input_example.yaml` before comparing
Q or tau_E with the paper: the published scalars are not mutually consistent
under a plain 0-D energy balance, and `H_NT23` has a power degradation
exponent of -0.89, which makes `P_heat = (W_tot/K)**9.09` and therefore very
stiff. This is a scoping model of the CENTAUR machine, not a reproduction of
the CENTAUR design point.

In [ ]:
i = int(np.abs(pc.output.n_G_frac.values - 0.55).argmin())
j = int(np.abs(pc.output.T_i_avg.values - 6.5).argmin())
point = pc.output.isel(n_index=i, T_index=j)

print(f"n/n_G   = {float(point.n_G_frac):.2f}")
print(f"<T_i>   = {float(point.T_i_avg):.1f} keV")
print(f"P_fus   = {float(point.Pfusion):.0f} MW")
print(f"P_aux   = {float(point.Paux):.0f} MW")
print(f"Q       = {float(point.Q):.2f}")
print(f"beta_N  = {float(point.betaN):.2f}   (Table 1 gives 1.5)")

## Scoping a single operating point

`single_point` solves one density/temperature pair and shows the profiles
behind it, which is the quickest way to see why a point on the POPCON sits
where it does.

In [ ]:
pc.single_point(n_G_frac=0.55, Ti_av=6.5)

## Scanning the current against the field

This is the main worked scan in the repository. `POPCON_scan` runs a full
POPCON at every combination of two machine parameters and tiles them. The
3 x 3 grid of `I_P` against `B_0` below comes from the `scan:` block in the
settings file.

Only parameters that are re-derived on every run can be scanned; the list is
in `openpopcon.SCANNABLE_SETTINGS_KEYS`. Scanning geometry on an example that
supplies a gEQDSK is refused, because the equilibrium overrides it.

In [ ]:
print(sorted(op.SCANNABLE_SETTINGS_KEYS))

In [ ]:
sc = op.POPCON_scan(settingsfile=settingsfile, plotsettingsfile=plotsettingsfile)
sc.run_scan()

Every panel is put on the same contour levels, so they can be compared
directly. The y axis is shared because it is the Greenwald fraction; had the
plotsettings asked for an absolute density it would not be, since
`n_G = I_p / (pi a^2)` moves with the scanned current.

In [ ]:
fig, axs = sc.plot()

In [ ]:
fig, ax = sc.plot_metric("Q", reduce="max")

### Scanning from Python instead of the settings file

Passing `scan=` overrides the block in the settings file. Each axis takes
either an explicit list of values or a `min`/`max`/`N` range.

In [ ]:
sc2 = op.POPCON_scan(
    settingsfile=settingsfile,
    plotsettingsfile=plotsettingsfile,
    scan={
        "rows": ("H_fac", [0.7, 0.87, 1.0]),
        "cols": ("delta", {"min": -0.55, "max": -0.25, "N": 3}),
    },
)
sc2.run_scan()
fig, axs = sc2.plot()

## Saving and reloading a scan

`write_output` saves the whole scan: the base settings, the scan
specification, the combined arrays and the grid plot.

In [ ]:
sc.write_output(name="centaur_scan", archive=False, overwrite=True)

back = op.POPCON_scan.read_output("centaur_scan")
print(back.shape, back.row.parameter, back.col.parameter)